# Memory Capacity (MC) Evaluation of Echo State Networks 🧠

This notebook evaluates the Memory Capacity (MC) of an Echo State Network (ESN) using different connectome matrices as the reservoir. It replicates the methodology described in the paper *Brain connectivity meets reservoir computing* by Damicelli et al. (2022).

## 1. Setup

First, let's install and import the necessary libraries. We'll be using `echoes` for the ESN implementation, `numpy` for numerical operations, `matplotlib` for plotting, and `bio2art` to load the example connectomes.

In [ ]:
# !pip install -q echoes bio2art numpy matplotlib scipy

In [1]:
import echoes
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os
import urllib.request
import zipfile

print(f"Using echoes version: {echoes.__version__}")

Using echoes version: 1.0.2


## 2. Load Connectomes

### Damicelli's Example Connectomes

The connectomes used in the paper (Macaque, Marmoset, and Human) are available in the `bio2art` GitHub repository. The following code will download the repository and load the connectome matrices.

In [6]:
import echoes.datasets
import bio2art.connectomes

ex_conn_mackeyglasst17 = echoes.datasets.load_mackeyglasst17()
ex_conn_human = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/X_notebooks/bio2art-master/bio2art/connectomes/C_Human_Betzel_Normalized.npy")


In [12]:
def download_and_unzip(url, extract_to='.'):
    zip_path, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(extract_to)
    os.remove(zip_path)

if not os.path.exists('bio2art-master'):
    print("Downloading bio2art repository...")
    download_and_unzip('https://github.com/AlGoulas/bio2art/archive/refs/heads/master.zip')
    print("Download complete.")
else:
    print("bio2art repository already downloaded.")

# Load the connectomes
data_path = '/Users/adrian/Documents/01_projects/14_4D_lab/X_notebooks/bio2art-master/bio2art/connectomes/'
macaque_connectome = np.load(os.path.join(data_path, 'ND_Macaque_Normalized.npy'))
# marmoset_connectome = np.load(os.path.join(data_path, 'marmoset.npy'))
# human_connectome = np.load(os.path.join(data_path, 'human.npy'))

example_connectomes = {
    "Macaque": macaque_connectome,
    # "Marmoset": marmoset_connectome,
    # "Human": human_connectome
}

print("Example connectomes loaded:")
for name, conn in example_connectomes.items():
    print(f"- {name}: shape {conn.shape}")

bio2art repository already downloaded.
Example connectomes loaded:
- Macaque: shape (29,)


### Load Your Connectome

Now, you can load your own connectome. **Replace `'YOUR_CONNECTOME_FILE.npy'` with the path to your file.**

In [13]:
# <<< REPLACE THIS WITH THE PATH TO YOUR CONNECTOME FILE >>>
user_connectome_path = '/Users/adrian/Documents/01_projects/14_4D_lab/OLD_output_8_working_files/00_gnm_experiments_20250912_122051/generated_networks/net_eta-0.772_gamma0.192_ruleMatchingIndex.npy' 

user_connectome = None
if os.path.exists(user_connectome_path):
    user_connectome = np.load(user_connectome_path)
    print(f"Successfully loaded your connectome from {user_connectome_path} with shape {user_connectome.shape}")
else:
    print("User connectome file not found. Please update the 'user_connectome_path' variable.")

Successfully loaded your connectome from /Users/adrian/Documents/01_projects/14_4D_lab/OLD_output_8_working_files/00_gnm_experiments_20250912_122051/generated_networks/net_eta-0.772_gamma0.192_ruleMatchingIndex.npy with shape (100, 68, 68)


## 3. Memory Capacity (MC) Evaluation

Here, we'll define a function to evaluate the MC of a given connectome. This function will:

1.  **Generate the MC task data**: A random input signal `X` and a series of time-lagged versions of `X` as the target `y`.
2.  **Create and train an ESN**: The provided connectome will be used as the reservoir's weight matrix `W`.
3.  **Calculate the MC score**: The score is the sum of the squared Pearson correlation coefficients between the network's output and the target for each time lag.

In [63]:
from src.ESNs.alternative_test_memory_capacity_weighted import alternative_evaluate_mc

In [ ]:
# def evaluate_mc(W, n_lags=50, train_len=4000, test_len=1000):
#     """Evaluates the Memory Capacity of a given reservoir matrix W."""
    
#     # 1. Generate data for the MC task
#     random_sequence = np.random.uniform(-0.5, 0.5, train_len + test_len)
#     X = random_sequence.reshape(-1, 1)
#     y = np.zeros((len(X), n_lags))
#     for i in range(1, n_lags + 1):
#         y[i:, i - 1] = X[:-i, 0]
    
#     X_train, X_test = X[:train_len], X[train_len:]
#     y_train, y_test = y[:train_len], y[train_len:]

#     # 2. Create and train the ESN
#     esn = echoes.ESNRegressor(
#         W=W, 
#         # spectral_radius=0.99, 
#         # input_scaling=1e-5, 
#         # leak_rate=1, 
#         # bias=1,
#         # random_state=42
#     )
    
#     esn.fit(X_train, y_train)
#     y_pred = esn.predict(X_test)

#     # 3. Calculate the MC score
#     mc_score = 0
#     for i in range(n_lags):
#         corr, _ = pearsonr(y_test[100:, i], y_pred[100:, i]) # Discard initial transient
#         mc_score += corr**2
        
#     return mc_score

## 4. Run Evaluation and Display Results

Now we'll run the evaluation for the example connectomes and your provided connectome, then visualize the results.

In [61]:
ex_conn = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/X_notebooks/bio2art-master/bio2art/connectomes/C_Marmoset_Normalized.npy")[:19,:19]
# ex_conn = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/X_notebooks/bio2art-master/bio2art/connectomes/C_Mouse_Gamanut_Normalized.npy")
print(ex_conn.shape)
# set random connections in ex_conn to zero 
# Set the desired sparsity level (e.g., 0.2 for 20%)
sparsity_level = 0.009

# Create a random mask and set connections to zero
random_mask = np.random.rand(*ex_conn.shape) < sparsity_level
ex_conn[random_mask] = 0

(19, 19)


In [64]:
mc_scores = {}

# Evaluate example connectomes
# print("Evaluating example connectomes...")
# for name, conn in example_connectomes.items():
#     print(conn.shape)
#     score = evaluate_mc(conn)
#     mc_scores[name] = score
#     print(f"- MC for {name}: {score:.4f}")
    
# Evaluate user's connectome
if user_connectome is not None:
    print("\nEvaluating your connectome...")
    user_score = alternative_evaluate_mc(ex_conn) # user_connectome[0,:,:])
    print(user_score)
    mc_scores['Your Connectome'] = user_score
    print(f"- MC for Your Connectome: {user_score:.4f}")

# Visualize the results
# plt.figure(figsize=(10, 6))
# plt.bar(mc_scores.keys(), mc_scores.values(), color=['#ff9999','#66b3ff','#99ff99', '#ffcc99'])
# plt.title('Memory Capacity (MC) of Different Connectomes')
# plt.ylabel('Memory Capacity Score')
# plt.xlabel('Connectome')
# plt.xticks(rotation=15)
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.show()


Evaluating your connectome...
5.045668831390953
- MC for Your Connectome: 5.0457


/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/numpy/linalg/_linalg.py:3437: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/echoes/esn/_base.py:312: RuntimeWarning: divide by zero encountered in matmul
  W_out = (np.linalg.pinv(full_states) @ outputs).T
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/echoes/esn/_base.py:312: RuntimeWarning: overflow encountered in matmul
  W_out = (np.linalg.pinv(full_states) @ outputs).T
/opt/miniconda3/envs/ma_thesis/lib/python3.13/site-packages/echoes/esn/_base.py:312: RuntimeWarning: invalid value